[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/03_ONNX_Architecture_and_Internals/04_Type_System_and_Shapes/Type_System_and_Shapes_Deep_Dive.ipynb)

# 3.4 Type System and Shapes — Deep Dive

ONNX's type system governs **what data flows through the graph** and **what shapes those tensors have**. This section formalizes the type constraint system, shape inference algorithm, and broadcasting rules that underpin every ONNX model.

---

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [The ONNX Type Universe](#section-1) | Element types, tensor types, sequence/map/optional |
| 2 | [Type Constraints — Parametric Polymorphism](#section-2) | Type variables, binding rules |
| 3 | [Shape Representation](#section-3) | Static, symbolic, and unknown dimensions |
| 4 | [Shape Inference Algorithm](#section-4) | How shapes propagate through graphs |
| 5 | [Broadcasting Rules](#section-5) | NumPy broadcasting formalized |
| 6 | [Shape Inference for Common Operators](#section-6) | Per-operator rules |
| 7 | [Type and Shape Inference in Practice](#section-7) | Code walkthrough |
| 8 | [Edge Cases and Limitations](#section-8) | When inference fails |
| 9 | [Key Takeaways & Interview Questions](#section-9) | Summary |

### Prerequisites

- Completed **3.1–3.3** (Computation Graphs, Nodes, IR Specification)
- Familiarity with NumPy broadcasting

<a id='section-1'></a>
## Section 1: The ONNX Type Universe

### Type Hierarchy

```
TypeProto
├── tensor_type       : Tensor<elem_type, shape>
│   ├── elem_type     : DataType enum (FLOAT, INT32, ...)
│   └── shape         : TensorShapeProto
│       └── dim[]     : { dim_value | dim_param }
├── sequence_type     : Sequence<elem_type>
│   └── elem_type     : TypeProto (recursive)
├── map_type          : Map<key_type, value_type>
│   ├── key_type      : DataType
│   └── value_type    : TypeProto
├── optional_type     : Optional<elem_type>
│   └── elem_type     : TypeProto
└── sparse_tensor_type : SparseTensor<elem_type, shape>
```

### Element Types

| Category | Types | Usage |
|----------|-------|-------|
| **Floating point** | float16, float32, float64, bfloat16 | Activations, weights |
| **Signed integer** | int8, int16, int32, int64 | Indices, quantized values |
| **Unsigned integer** | uint8, uint16, uint32, uint64 | Quantized activations |
| **Boolean** | bool | Masks, conditions |
| **String** | string | Text data |
| **Complex** | complex64, complex128 | Signal processing |

### Formal Type Notation

We write ONNX types as:

$$\text{Tensor}\langle \tau, [d_1, d_2, \ldots, d_r] \rangle$$

where $\tau$ is the element type and $d_i$ are dimension specifications. Each $d_i$ is one of:

- **Static**: a concrete integer (e.g., $d_i = 768$)
- **Symbolic**: a named parameter (e.g., $d_i = \text{"batch"}$)
- **Unknown**: unspecified (e.g., $d_i = ?$)

<a id='section-2'></a>
## Section 2: Type Constraints — Parametric Polymorphism

### The Problem

ONNX operators work with multiple data types. The `Add` operator can add float32 tensors, float64 tensors, int32 tensors, etc. Without a type system, we would need separate operators: `AddFloat32`, `AddFloat64`, `AddInt32`, ...

### Solution: Type Variables

ONNX uses **type variables** (type parameters) constrained to allowed sets:

$$\text{Add}(A: T, B: T) \to T$$
$$\text{where } T \in \{\text{float16}, \text{float32}, \text{float64}, \text{int32}, \text{int64}, \ldots\}$$

### Binding Rules

All occurrences of the same type variable $T$ in a single node must bind to the **same concrete type**:

$$\text{If } A \in \text{Tensor}\langle\text{float32}\rangle \text{ then } T = \text{float32}$$
$$\implies B \text{ must also be } \text{Tensor}\langle\text{float32}\rangle$$
$$\implies \text{output must be } \text{Tensor}\langle\text{float32}\rangle$$

### Multiple Type Variables

Some operators have multiple independent type variables:

$$\text{Cast}(\text{input}: T_1) \to T_2$$
$$\text{where } T_1 \in \{\text{all numeric}\}, \quad T_2 \in \{\text{all numeric}\}$$

Here $T_1$ and $T_2$ can bind to different types (that's the whole point of Cast).

### Type Checking Algorithm

```
TYPE_CHECK(node, input_types):
  1. schema ← GET_SCHEMA(node.op_type, node.domain)
  2. bindings ← {}  (type variable → concrete type)
  3. For each (formal_input, actual_type) in zip(schema.inputs, input_types):
     a. tv ← formal_input.type_variable  (e.g., "T")
     b. If tv in bindings:
          ASSERT bindings[tv] == actual_type  (consistency)
        Else:
          ASSERT actual_type in schema.constraints[tv]
          bindings[tv] ← actual_type
  4. For each formal_output:
     output_type ← bindings[formal_output.type_variable]
  5. Return output_types
```

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install onnx onnxruntime numpy matplotlib

In [ ]:
import numpy as np
import onnx
from onnx import TensorProto, helper, numpy_helper, defs
from onnx.checker import check_model
from onnx.shape_inference import infer_shapes
import onnxruntime as ort

# Demonstrate type constraints
def show_type_constraints(op_name, domain=''):
    schema = defs.get_schema(op_name, domain=domain)
    print(f'{op_name} Type Constraints:')
    for tc in schema.type_constraints:
        types = sorted(str(t) for t in tc.allowed_type_strs)
        print(f'  {tc.type_param_str}: {types[:6]}' +
              (f' ... ({len(types)} total)' if len(types) > 6 else ''))
    print(f'  Signature: {op_name}(' +
          ', '.join(f'{i.name}: {i.typeStr}' for i in schema.inputs) + ') → ' +
          ', '.join(f'{o.name}: {o.typeStr}' for o in schema.outputs))
    print()

show_type_constraints('Add')
show_type_constraints('MatMul')
show_type_constraints('Cast')
show_type_constraints('Where')

In [ ]:
# Demonstrate type constraint violation
# Try to Add a float32 and int32 tensor
X_f = helper.make_tensor_value_info('X_f', TensorProto.FLOAT, [2, 3])
X_i = helper.make_tensor_value_info('X_i', TensorProto.INT32, [2, 3])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [2, 3])

node = helper.make_node('Add', ['X_f', 'X_i'], ['Y'])
graph = helper.make_graph([node], 'type_error', [X_f, X_i], [Y])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 18)])

try:
    check_model(model)
    print('Model passed validation (unexpected)')
except onnx.checker.ValidationError as e:
    error_msg = str(e)[:200]
    print(f'Type constraint violation detected:')
    print(f'  {error_msg}')
    print(f'\n  Reason: Add requires both inputs to share the same')
    print(f'  type variable T, but X_f is FLOAT and X_i is INT32.')

<a id='section-3'></a>
## Section 3: Shape Representation

### Dimension Types

| Type | Protobuf Field | Example | Meaning |
|------|:---------------|:--------|:--------|
| **Static** | `dim_value` | `768` | Fixed at graph construction |
| **Symbolic** | `dim_param` | `"batch"` | Named variable, resolved at runtime |
| **Unknown** | neither set | — | Completely unknown |

### Shape Notation

We denote shapes using brackets with the dimension types:

$$[\text{batch}, 768] \quad \text{means dim 0 is symbolic, dim 1 is static}$$
$$[N, C, H, W] \quad \text{batch, channels, height, width (all symbolic)}$$
$$[?, ?, ?] \quad \text{rank=3, all dimensions unknown}$$

### Shape Compatibility

Two shapes are **compatible** if they can be unified:

$$\text{compatible}(S_1, S_2) \iff \text{rank}(S_1) = \text{rank}(S_2) \wedge \forall i: \text{unify}(S_1[i], S_2[i]) \neq \bot$$

where unification rules are:

$$\text{unify}(n, n) = n \quad \text{(same static)}$$
$$\text{unify}(n, ?) = n \quad \text{(static wins)}$$
$$\text{unify}(s, ?) = s \quad \text{(symbolic wins)}$$
$$\text{unify}(s, s) = s \quad \text{(same symbol)}$$
$$\text{unify}(n, m) = \bot \quad \text{if } n \neq m \; \text{(conflict!)}$$

In [ ]:
# Demonstrate different shape specifications
shapes = [
    ('Static', [32, 768]),
    ('Symbolic', ['batch', 768]),
    ('Mixed', ['batch', 'seq_len', 768]),
    ('Unknown rank', None),
]

print('Shape Specification Examples')
print('=' * 60)
for name, shape in shapes:
    vi = helper.make_tensor_value_info(f'x_{name}', TensorProto.FLOAT, shape)
    tt = vi.type.tensor_type
    if tt.HasField('shape'):
        dims = []
        for d in tt.shape.dim:
            if d.dim_param:
                dims.append(f'"{d.dim_param}"')
            elif d.dim_value:
                dims.append(str(d.dim_value))
            else:
                dims.append('?')
        print(f'  {name:15s} → [{', '.join(dims)}]')
    else:
        print(f'  {name:15s} → (no shape - unknown rank)')

<a id='section-4'></a>
## Section 4: Shape Inference Algorithm

### What is Shape Inference?

Shape inference propagates type and shape information from graph inputs through all nodes to compute the type and shape of every intermediate tensor.

### The Algorithm

$$\text{INFER\_SHAPES}(G):$$

```
SHAPE_INFERENCE(graph):
  1. known ← {}  (tensor_name → (type, shape))

  2. For each input i in graph.input:
       known[i.name] ← (i.type, i.shape)

  3. For each initializer w in graph.initializer:
       known[w.name] ← (w.data_type, w.dims)

  4. For each node v in topological order:
       input_shapes ← [known[name] for name in v.input]
       output_shapes ← INFER_NODE(v.op_type, input_shapes, v.attributes)
       for name, shape in zip(v.output, output_shapes):
         known[name] ← shape
         graph.value_info.add(name, shape)  ← store result

  5. Return graph
```

### Per-Operator Inference

Each operator defines its own shape inference function. For example:

$$\text{INFER}_{\text{MatMul}}([m, k], [k, n]) = [m, n]$$
$$\text{INFER}_{\text{Relu}}([d_1, \ldots, d_r]) = [d_1, \ldots, d_r]$$
$$\text{INFER}_{\text{Concat}}(\{S_i\}, \text{axis}=a) = [d_1, \ldots, \sum_i S_i[a], \ldots, d_r]$$

### Visualization

```
Before Shape Inference:           After Shape Inference:

X: [batch, 4]                     X: [batch, 4]
     │                                 │
  MatMul ──▶ h1: ???              MatMul ──▶ h1: [batch, 8]
     │                                 │
   Add   ──▶ h1b: ???              Add   ──▶ h1b: [batch, 8]
     │                                 │
   Relu  ──▶ a1: ???              Relu  ──▶ a1: [batch, 8]
     │                                 │
  MatMul ──▶ h2: ???              MatMul ──▶ h2: [batch, 2]
     │                                 │
   Add   ──▶ Y: [batch, 2]         Add   ──▶ Y: [batch, 2]
```

In [ ]:
# Build a model and run shape inference
rng = np.random.default_rng(42)

W1 = numpy_helper.from_array(
    rng.standard_normal((4, 8)).astype(np.float32), name='W1')
b1 = numpy_helper.from_array(
    rng.standard_normal((8,)).astype(np.float32), name='b1')
W2 = numpy_helper.from_array(
    rng.standard_normal((8, 2)).astype(np.float32), name='W2')
b2 = numpy_helper.from_array(
    rng.standard_normal((2,)).astype(np.float32), name='b2')

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, ['batch', 4])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, ['batch', 2])

nodes = [
    helper.make_node('MatMul', ['X', 'W1'], ['h1'], name='fc1'),
    helper.make_node('Add', ['h1', 'b1'], ['h1b'], name='bias1'),
    helper.make_node('Relu', ['h1b'], ['a1'], name='relu1'),
    helper.make_node('MatMul', ['a1', 'W2'], ['h2'], name='fc2'),
    helper.make_node('Add', ['h2', 'b2'], ['Y'], name='bias2'),
]

graph = helper.make_graph(nodes, 'shape_demo', [X], [Y], [W1, b1, W2, b2])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 18)])
check_model(model)

# Before shape inference
print('BEFORE Shape Inference:')
print(f'  value_info count: {len(model.graph.value_info)}')
print(f'  (intermediate shapes unknown)\n')

# Run shape inference
model_inferred = infer_shapes(model)

# After shape inference
print('AFTER Shape Inference:')
print(f'  value_info count: {len(model_inferred.graph.value_info)}')
print(f'\n  Intermediate tensor shapes:')
for vi in model_inferred.graph.value_info:
    tt = vi.type.tensor_type
    dtype = TensorProto.DataType.Name(tt.elem_type)
    dims = [d.dim_param or d.dim_value for d in tt.shape.dim]
    print(f'    {vi.name:8s}: {dtype} {dims}')

print(f'\n  Graph output shapes:')
for out in model_inferred.graph.output:
    tt = out.type.tensor_type
    dtype = TensorProto.DataType.Name(tt.elem_type)
    dims = [d.dim_param or d.dim_value for d in tt.shape.dim]
    print(f'    {out.name:8s}: {dtype} {dims}')

<a id='section-5'></a>
## Section 5: Broadcasting Rules

### Definition

ONNX follows **NumPy broadcasting rules** for element-wise operations. Given two tensors with shapes $A$ and $B$:

### Step-by-Step Broadcasting Algorithm

**Step 1**: Right-align the shapes and pad the shorter one with 1s on the left:

$$A = [d_1^A, d_2^A, \ldots, d_r^A], \quad B = [d_1^B, d_2^B, \ldots, d_s^B]$$

If $r < s$, pad $A$: $A' = [\underbrace{1, \ldots, 1}_{s-r}, d_1^A, \ldots, d_r^A]$

**Step 2**: For each dimension $i$, the output dimension is:

$$d_i^{\text{out}} = \begin{cases} d_i^A & \text{if } d_i^A = d_i^B \\ \max(d_i^A, d_i^B) & \text{if } d_i^A = 1 \text{ or } d_i^B = 1 \\ \bot & \text{otherwise (error)} \end{cases}$$

### Broadcasting Examples

```
   Shape A       Shape B       Result       Rule Applied
   ───────       ───────       ──────       ────────────
   [5, 3]    +   [5, 3]    =  [5, 3]       exact match
   [5, 3]    +   [   3]    =  [5, 3]       B padded to [1,3], dim 0 broadcast
   [5, 1]    +   [1, 3]    =  [5, 3]       both dims broadcast
   [5, 3]    +   [   1]    =  [5, 3]       scalar broadcast
   [N, D]    +   [   D]    =  [N, D]       bias addition pattern
   [5, 3]    +   [4, 3]    =  ERROR        5 ≠ 4, neither is 1
```

In [ ]:
# Demonstrate broadcasting in ONNX
def test_broadcast(shape_a, shape_b, label=''):
    """Build and run an Add model to verify broadcasting."""
    A = helper.make_tensor_value_info('A', TensorProto.FLOAT, list(shape_a))
    B = helper.make_tensor_value_info('B', TensorProto.FLOAT, list(shape_b))
    C = helper.make_tensor_value_info('C', TensorProto.FLOAT, None)

    node = helper.make_node('Add', ['A', 'B'], ['C'])
    graph = helper.make_graph([node], 'broadcast_test', [A, B], [C])
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 18)])

    model_inf = infer_shapes(model)
    out_shape = [d.dim_value for d in
                 model_inf.graph.output[0].type.tensor_type.shape.dim]

    a = np.ones(shape_a, dtype=np.float32)
    b = np.ones(shape_b, dtype=np.float32)
    sess = ort.InferenceSession(model.SerializeToString(),
                                providers=['CPUExecutionProvider'])
    result = sess.run(None, {'A': a, 'B': b})[0]

    print(f'  {str(list(shape_a)):12s} + {str(list(shape_b)):12s} '
          f'→ inferred={out_shape}  actual={list(result.shape)}  '
          f'{label}')

print('Broadcasting Verification')
print('=' * 70)
test_broadcast((5, 3), (5, 3), 'exact match')
test_broadcast((5, 3), (3,), 'row broadcast')
test_broadcast((5, 1), (1, 3), 'both broadcast')
test_broadcast((5, 3), (1,), 'scalar broadcast')
test_broadcast((2, 3, 4), (4,), '3D + 1D')
test_broadcast((2, 3, 4), (3, 1), '3D + 2D')

<a id='section-6'></a>
## Section 6: Shape Inference for Common Operators

### Operator-Specific Rules

#### MatMul

For $A \in \mathbb{R}^{\ldots \times m \times k}$ and $B \in \mathbb{R}^{\ldots \times k \times n}$:

$$\text{shape}(\text{MatMul}(A, B)) = [\text{broadcast}(\text{batch\_dims}), m, n]$$

The inner dimensions must match: $A.\text{shape}[-1] = B.\text{shape}[-2]$

#### Reshape

The output shape is specified by the `shape` input, subject to the constraint:

$$\prod_{i} d_i^{\text{out}} = \prod_{j} d_j^{\text{in}}$$

A single dimension can be $-1$ (inferred):

$$d_{-1} = \frac{\prod_j d_j^{\text{in}}}{\prod_{i \neq -1} d_i^{\text{out}}}$$

#### Conv

For input $X: [N, C_{\text{in}}, H, W]$ and kernel $K: [C_{\text{out}}, C_{\text{in}}/g, k_h, k_w]$:

$$H_{\text{out}} = \left\lfloor \frac{H + 2p_h - k_h}{s_h} \right\rfloor + 1$$
$$W_{\text{out}} = \left\lfloor \frac{W + 2p_w - k_w}{s_w} \right\rfloor + 1$$
$$\text{Output}: [N, C_{\text{out}}, H_{\text{out}}, W_{\text{out}}]$$

#### Transpose

Given permutation $\pi$:

$$\text{shape}(\text{Transpose}(X, \text{perm}=\pi)) = [d_{\pi(0)}, d_{\pi(1)}, \ldots, d_{\pi(r-1)}]$$

In [ ]:
# Verify shape inference rules for multiple operators
def infer_and_show(name, nodes, inputs, outputs, initializers=None):
    """Build model, run shape inference, show results."""
    graph = helper.make_graph(
        nodes, name, inputs, outputs, initializers or [])
    model = helper.make_model(
        graph, opset_imports=[helper.make_opsetid('', 18)])
    model_inf = infer_shapes(model)

    print(f'\n  {name}:')
    for vi in model_inf.graph.value_info:
        tt = vi.type.tensor_type
        dims = [d.dim_param or d.dim_value for d in tt.shape.dim]
        print(f'    {vi.name}: {dims}')
    for out in model_inf.graph.output:
        tt = out.type.tensor_type
        dims = [d.dim_param or d.dim_value for d in tt.shape.dim]
        print(f'    {out.name}: {dims}  (output)')

print('Shape Inference Verification')
print('=' * 60)

# MatMul: [batch, 4] x [4, 8] → [batch, 8]
W_init = numpy_helper.from_array(
    np.zeros((4, 8), dtype=np.float32), name='W')
infer_and_show('MatMul', [
    helper.make_node('MatMul', ['X', 'W'], ['Y']),
], [helper.make_tensor_value_info('X', TensorProto.FLOAT, ['batch', 4])],
   [helper.make_tensor_value_info('Y', TensorProto.FLOAT, None)],
   [W_init])

# Transpose: [2, 3, 4] with perm=[2,0,1] → [4, 2, 3]
infer_and_show('Transpose', [
    helper.make_node('Transpose', ['X'], ['Y'], perm=[2, 0, 1]),
], [helper.make_tensor_value_info('X', TensorProto.FLOAT, [2, 3, 4])],
   [helper.make_tensor_value_info('Y', TensorProto.FLOAT, None)])

# Concat: [2, 3] + [2, 5] along axis=1 → [2, 8]
infer_and_show('Concat', [
    helper.make_node('Concat', ['A', 'B'], ['Y'], axis=1),
], [helper.make_tensor_value_info('A', TensorProto.FLOAT, [2, 3]),
    helper.make_tensor_value_info('B', TensorProto.FLOAT, [2, 5])],
   [helper.make_tensor_value_info('Y', TensorProto.FLOAT, None)])

<a id='section-7'></a>
## Section 7: Type and Shape Inference in Practice

In [ ]:
# Full end-to-end: build → infer → verify → run
rng = np.random.default_rng(0)

# Build a 3-layer MLP with symbolic batch dimension
layers = [(4, 16, 'layer1'), (16, 8, 'layer2'), (8, 2, 'layer3')]
all_inits = []
all_nodes = []
prev_out = 'X'

for in_dim, out_dim, name in layers:
    w_name = f'W_{name}'
    b_name = f'b_{name}'
    mm_out = f'mm_{name}'
    add_out = f'add_{name}'
    act_out = f'act_{name}' if name != 'layer3' else 'Y'

    all_inits.append(numpy_helper.from_array(
        rng.standard_normal((in_dim, out_dim)).astype(np.float32), name=w_name))
    all_inits.append(numpy_helper.from_array(
        rng.standard_normal((out_dim,)).astype(np.float32), name=b_name))

    all_nodes.append(helper.make_node('MatMul', [prev_out, w_name], [mm_out]))
    all_nodes.append(helper.make_node('Add', [mm_out, b_name], [add_out]))
    if name != 'layer3':
        all_nodes.append(helper.make_node('Relu', [add_out], [act_out]))
    prev_out = act_out

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, ['batch', 4])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, ['batch', 2])

graph = helper.make_graph(all_nodes, 'mlp_3layer', [X], [Y], all_inits)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 18)])
check_model(model)

# Shape inference
model_inf = infer_shapes(model)

print('Three-Layer MLP — Shape Propagation')
print('=' * 60)
print(f'\nInput:')
for inp in model_inf.graph.input:
    if inp.name == 'X':
        tt = inp.type.tensor_type
        dims = [d.dim_param or d.dim_value for d in tt.shape.dim]
        print(f'  {inp.name}: {dims}')

print(f'\nIntermediate shapes (from shape inference):')
for vi in model_inf.graph.value_info:
    tt = vi.type.tensor_type
    dtype = TensorProto.DataType.Name(tt.elem_type)
    dims = [d.dim_param or d.dim_value for d in tt.shape.dim]
    print(f'  {vi.name:15s}: {dtype} {dims}')

print(f'\nOutput:')
for out in model_inf.graph.output:
    tt = out.type.tensor_type
    dims = [d.dim_param or d.dim_value for d in tt.shape.dim]
    print(f'  {out.name}: {dims}')

# Verify with actual inference
sess = ort.InferenceSession(model.SerializeToString(),
                            providers=['CPUExecutionProvider'])
for batch_size in [1, 5, 32]:
    x_test = rng.standard_normal((batch_size, 4)).astype(np.float32)
    result = sess.run(None, {'X': x_test})[0]
    print(f'  batch={batch_size:3d} → output shape: {result.shape}')

<a id='section-8'></a>
## Section 8: Edge Cases and Limitations

### When Shape Inference Fails

| Scenario | Example | Result |
|----------|---------|--------|
| **Data-dependent shapes** | `NonZero(X)` | Output shape depends on values of X |
| **Dynamic Reshape** | `Reshape(X, shape_input)` | Shape comes from a runtime tensor |
| **Custom operators** | `CustomOp(X)` | No schema → no inference rule |
| **Symbolic conflicts** | `[N] + [M]` where N ≠ M | Cannot determine without runtime values |

### Data-Dependent Shapes

Some operators produce outputs whose shapes depend on input **values** (not just input shapes):

$$\text{NonZero}(X \in \mathbb{R}^{d_1 \times \ldots \times d_r}) \to \mathbb{Z}^{r \times k}$$

where $k = |\{i : X_i \neq 0\}|$ — the number of non-zero elements, which depends on the **values** of $X$.

### Best Practices

1. Always run `infer_shapes()` after building a model
2. Use symbolic dimensions instead of `None` when possible
3. Share symbolic names across related dimensions (e.g., same `"batch"` for all batch dims)
4. Check `value_info` to verify inference succeeded for intermediate tensors

In [ ]:
# Demonstrate data-dependent shape
X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [3, 4])
Y = helper.make_tensor_value_info('Y', TensorProto.INT64, None)

node = helper.make_node('NonZero', ['X'], ['Y'])
graph = helper.make_graph([node], 'nonzero_demo', [X], [Y])
model_nz = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 18)])
model_nz_inf = infer_shapes(model_nz)

out_type = model_nz_inf.graph.output[0].type.tensor_type
if out_type.HasField('shape'):
    dims = []
    for d in out_type.shape.dim:
        if d.dim_param:
            dims.append(d.dim_param)
        elif d.dim_value:
            dims.append(d.dim_value)
        else:
            dims.append('?')
    print(f'NonZero inferred shape: {dims}')
else:
    print('NonZero: shape unknown (data-dependent)')

# Show that runtime shape varies
sess = ort.InferenceSession(model_nz.SerializeToString(),
                            providers=['CPUExecutionProvider'])
x_dense = np.ones((3, 4), dtype=np.float32)
x_sparse = np.array([[1, 0, 0, 2], [0, 0, 0, 0], [3, 0, 4, 0]], dtype=np.float32)

print(f'\nRuntime shape variation:')
for label, x in [('dense (all nonzero)', x_dense), ('sparse', x_sparse)]:
    result = sess.run(None, {'X': x})[0]
    print(f'  {label:25s}: NonZero output shape = {result.shape}')

<a id='section-9'></a>
## Section 9: Key Takeaways & Interview Questions

### Summary

| Concept | Key Point |
|---------|----------|
| **Type universe** | Tensor, Sequence, Map, Optional, SparseTensor |
| **Type constraints** | Parametric polymorphism via type variables ($T$) |
| **Shape types** | Static (int), symbolic (string), unknown (unset) |
| **Shape inference** | Forward propagation in topological order |
| **Broadcasting** | NumPy rules: right-align, pad with 1s, expand |
| **Limitations** | Data-dependent shapes (NonZero, etc.) cannot be fully inferred |

### Interview Questions

1. **Q**: What is a type constraint in ONNX, and why is it needed?
   - **A**: A type constraint binds a type variable (like `T`) to a set of allowed types. All inputs/outputs sharing the same type variable must have the same concrete type. This enables a single operator definition (like `Add`) to work with float32, float64, int32, etc. without separate operator implementations.

2. **Q**: Explain the broadcasting rule for `Add([5, 1], [1, 3])`.
   - **A**: Shapes are already right-aligned and same rank. Dimension 0: 5 vs 1 → broadcast 1 to 5. Dimension 1: 1 vs 3 → broadcast 1 to 3. Result: `[5, 3]`.

3. **Q**: Why can't shape inference determine the output shape of `NonZero`?
   - **A**: `NonZero` returns the indices of non-zero elements, so its output shape depends on the **values** of the input tensor, not just its shape. Different input values produce different numbers of non-zero elements, making the output shape unknowable at graph construction time.

4. **Q**: What is the difference between a static dimension and a symbolic dimension?
   - **A**: A static dimension is a fixed integer (e.g., 768) known at graph construction. A symbolic dimension is a named variable (e.g., `"batch"`) whose concrete value is determined at runtime. Symbolic dimensions allow the same model to handle different input sizes while still enabling shape inference for related dimensions.

---

**Congratulations!** You have completed Module 3: ONNX Architecture and Internals. You now have a deep understanding of computation graphs, nodes, the IR specification, and the type system.